# Gold - ecommerce_rastreamento

Notebook para criação de tabelas Gold em Delta Lake e réplica opcional para SQL Server/Azure, mantendo padrão de consumo analítico via Looker.

Este notebook assume que as tabelas Silver e `squad1.dq_monitoring_logs` já foram criadas em Delta.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SCHEMA = "squad1"
DQ_LOGS_TABLE = f"{SCHEMA}.dq_monitoring_logs"

# Ative como True somente se desejar replicar as Golds para SQL Server/Azure.
REPLICAR_SQLSERVER = False

# Caso use SQL Server, o notebook config precisa ter carregado:
# JDBC_HOSTNAME, JDBC_DATABASE, JDBC_USERNAME, JDBC_PASSWORD


In [0]:
def tabela_delta_existe(nome_tabela: str) -> bool:
    return spark.catalog.tableExists(nome_tabela)


def salvar_gold_delta(df, tabela_destino: str, chaves_merge=None):
    """
    Salva a Gold como tabela Delta gerenciada.
    Para Gold agregada, usamos overwrite para recalcular o snapshot analítico.
    Isso evita duplicidade mesmo com múltiplas execuções.
    """
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela_destino)
    )
    print(f"Tabela Delta atualizada: {tabela_destino}")


def replicar_sqlserver(df, tabela_destino: str):
    """
    Replica para SQL Server usando overwrite, pois Gold agregada deve representar
    o estado analítico atual, não append incremental bruto.
    """
    if not REPLICAR_SQLSERVER:
        print(f"Réplica SQL Server desativada para {tabela_destino}")
        return

    (
        df.write
        .format("sqlserver")
        .mode("overwrite")
        .option("host", JDBC_HOSTNAME)
        .option("port", "1433")
        .option("database", JDBC_DATABASE)
        .option("user", JDBC_USERNAME)
        .option("password", JDBC_PASSWORD)
        .option("dbtable", tabela_destino)
        .option("encrypt", "true")
        .option("trustServerCertificate", "false")
        .save()
    )
    print(f"Tabela replicada para SQL Server: {tabela_destino}")


def criar_gold_dq_resumo_por_regra(nome_tabela_silver: str):
    df_logs = spark.table(DQ_LOGS_TABLE)

    return (
        df_logs
        .filter(F.col("tabela") == nome_tabela_silver)
        .withColumn("data_referencia", F.to_date("timestamp_execucao"))
        .groupBy("data_referencia", "tabela", "regra", "severidade")
        .agg(
            F.sum("qtd_registros_falhos").alias("qtd_registros_falhos"),
            F.sum("qtd_registros_total").alias("qtd_registros_total")
        )
        .withColumn(
            "perc_falha",
            F.when(F.col("qtd_registros_total") > 0,
                   F.round((F.col("qtd_registros_falhos") / F.col("qtd_registros_total")) * 100, 2))
             .otherwise(F.lit(0.0))
        )
        .withColumn("gold_processed_at", F.current_timestamp())
    )


def criar_gold_dq_resumo_por_tabela(df_silver, nome_tabela_silver: str, colunas_falha: list):
    cond_falha = None
    for c in colunas_falha:
        expr = F.coalesce(F.col(c), F.lit(False)) == True
        cond_falha = expr if cond_falha is None else (cond_falha | expr)

    return (
        df_silver
        .withColumn("hora_referencia", F.date_trunc("hour", F.col("silver_processed_at")))
        .withColumn("linha_com_falha", cond_falha)
        .groupBy("hora_referencia")
        .agg(
            F.count("*").alias("qtd_registros_total"),
            F.sum(F.when(F.col("linha_com_falha"), 1).otherwise(0)).alias("qtd_registros_com_falha")
        )
        .withColumn("tabela", F.lit(nome_tabela_silver))
        .withColumn("qtd_registros_limpos", F.col("qtd_registros_total") - F.col("qtd_registros_com_falha"))
        .withColumn(
            "perc_registros_limpos",
            F.when(F.col("qtd_registros_total") > 0,
                   F.round((F.col("qtd_registros_limpos") / F.col("qtd_registros_total")) * 100, 2))
             .otherwise(F.lit(0.0))
        )
        .withColumn("gold_processed_at", F.current_timestamp())
        .select(
            "hora_referencia", "tabela", "qtd_registros_total",
            "qtd_registros_com_falha", "qtd_registros_limpos",
            "perc_registros_limpos", "gold_processed_at"
        )
    )


##  Leitura das tabelas Silver e logs

In [0]:
SILVER_RASTREAMENTO = "squad1.silver_ecommerce_rastreamento"
SILVER_PEDIDOS = "squad1.silver_ecommerce_pedidos"

if not tabela_delta_existe(SILVER_RASTREAMENTO):
    raise Exception(f"Tabela não encontrada: {SILVER_RASTREAMENTO}")
if not tabela_delta_existe(DQ_LOGS_TABLE):
    raise Exception(f"Tabela não encontrada: {DQ_LOGS_TABLE}")

df_rastreamento = spark.table(SILVER_RASTREAMENTO)
df_pedidos = spark.table(SILVER_PEDIDOS) if tabela_delta_existe(SILVER_PEDIDOS) else None

display(df_rastreamento.limit(5))

##  Gold DQ - resumo por regra e por tabela

In [0]:
colunas_falha_rastreamento = [
    "r1_id_rastreamento_falhou",
    "r2_status_entrega_falhou",
    "r3_id_pedido_fk_falhou",
    "r4_dt_evento_falhou",
    "r5_codigo_rastreio_falhou",
    "r6_ordem_cronologica_falhou",
    "r7_entregue_sem_evento_falhou",
    "r8_sla_30_dias_falhou",
    "r9_cancelado_com_rastreamento_falhou",
    "r10_transportadora_inconsistente_falhou"
]
colunas_falha_rastreamento = [c for c in colunas_falha_rastreamento if c in df_rastreamento.columns]
if len(colunas_falha_rastreamento) == 0:
    raise Exception("Nenhuma coluna de falha encontrada na Silver de rastreamento.")

df_gold_dq_regra_rastreamento = criar_gold_dq_resumo_por_regra("silver_ecommerce_rastreamento")
df_gold_dq_tabela_rastreamento = criar_gold_dq_resumo_por_tabela(df_rastreamento, "silver_ecommerce_rastreamento", colunas_falha_rastreamento)

salvar_gold_delta(df_gold_dq_regra_rastreamento, "squad1.gold_rastreamento_dq_resumo_por_regra")
salvar_gold_delta(df_gold_dq_tabela_rastreamento, "squad1.gold_rastreamento_dq_resumo_por_tabela")
replicar_sqlserver(df_gold_dq_regra_rastreamento, "squad1.gold_rastreamento_dq_resumo_por_regra")
replicar_sqlserver(df_gold_dq_tabela_rastreamento, "squad1.gold_rastreamento_dq_resumo_por_tabela")

## KPIs específicos de rastreamento

In [0]:
df_rastreamento_base = (
    df_rastreamento
    .withColumn("dt_evento_ts", F.to_timestamp("dt_evento"))
    .withColumn("data_referencia", F.to_date("dt_evento"))
)

# Agrega datas por pedido para medir lead time coletado -> entregue
pivot_status = (
    df_rastreamento_base
    .groupBy("id_pedido_ecommerce")
    .agg(
        F.min(F.when(F.col("status_entrega") == "coletado", F.col("dt_evento_ts"))).alias("dt_coletado"),
        F.max(F.when(F.col("status_entrega") == "entregue", F.col("dt_evento_ts"))).alias("dt_entregue"),
        F.countDistinct("id_transportadora").alias("qtd_transportadoras"),
        F.max(F.when(F.col("status_entrega") == "entregue", 1).otherwise(0)).alias("tem_evento_entregue")
    )
    .withColumn("tempo_entrega_dias", F.datediff(F.col("dt_entregue"), F.col("dt_coletado")))
    .withColumn("sla_violado", F.col("tempo_entrega_dias") > 30)
)

df_gold_rastreamento_kpis = (
    df_rastreamento_base
    .groupBy("data_referencia")
    .agg(
        F.count("*").alias("qtd_eventos"),
        F.countDistinct("id_pedido_ecommerce").alias("qtd_pedidos_rastreados"),
        F.countDistinct("codigo_rastreio").alias("qtd_codigos_rastreio"),
        F.countDistinct("id_transportadora").alias("qtd_transportadoras"),
        F.sum(F.when(F.col("status_entrega") == "entregue", 1).otherwise(0)).alias("qtd_eventos_entregue")
    )
    .withColumn("gold_processed_at", F.current_timestamp())
)

salvar_gold_delta(df_gold_rastreamento_kpis, "squad1.gold_rastreamento_kpis")
replicar_sqlserver(df_gold_rastreamento_kpis, "squad1.gold_rastreamento_kpis")

display(df_gold_rastreamento_kpis)

##  SLA e performance por transportadora

In [0]:
df_gold_rastreamento_sla = (
    pivot_status
    .agg(
        F.count("*").alias("qtd_pedidos_rastreados"),
        F.sum(F.when(F.col("tem_evento_entregue") == 1, 1).otherwise(0)).alias("qtd_pedidos_entregues"),
        F.round(F.avg("tempo_entrega_dias"), 2).alias("tempo_medio_entrega_dias"),
        F.sum(F.when(F.col("sla_violado") == True, 1).otherwise(0)).alias("qtd_sla_violado")
    )
    .withColumn("perc_sla_violado", F.round((F.col("qtd_sla_violado") / F.col("qtd_pedidos_rastreados")) * 100, 2))
    .withColumn("gold_processed_at", F.current_timestamp())
)

salvar_gold_delta(df_gold_rastreamento_sla, "squad1.gold_rastreamento_sla")
replicar_sqlserver(df_gold_rastreamento_sla, "squad1.gold_rastreamento_sla")

# Performance por transportadora
transportadora_por_pedido = (
    df_rastreamento_base
    .groupBy("id_pedido_ecommerce", "id_transportadora")
    .agg(
        F.min(F.when(F.col("status_entrega") == "coletado", F.col("dt_evento_ts"))).alias("dt_coletado"),
        F.max(F.when(F.col("status_entrega") == "entregue", F.col("dt_evento_ts"))).alias("dt_entregue")
    )
    .withColumn("tempo_entrega_dias", F.datediff(F.col("dt_entregue"), F.col("dt_coletado")))
)

df_gold_rastreamento_transportadora = (
    transportadora_por_pedido
    .groupBy("id_transportadora")
    .agg(
        F.countDistinct("id_pedido_ecommerce").alias("qtd_pedidos"),
        F.round(F.avg("tempo_entrega_dias"), 2).alias("tempo_medio_entrega_dias"),
        F.sum(F.when(F.col("tempo_entrega_dias") > 30, 1).otherwise(0)).alias("qtd_sla_violado")
    )
    .withColumn("perc_sla_violado", F.round((F.col("qtd_sla_violado") / F.col("qtd_pedidos")) * 100, 2))
    .withColumn("gold_processed_at", F.current_timestamp())
)

salvar_gold_delta(df_gold_rastreamento_transportadora, "squad1.gold_rastreamento_por_transportadora")
replicar_sqlserver(df_gold_rastreamento_transportadora, "squad1.gold_rastreamento_por_transportadora")

In [0]:
tabelas_gold_criadas = ['squad1.gold_rastreamento_dq_resumo_por_regra', 'squad1.gold_rastreamento_dq_resumo_por_tabela', 'squad1.gold_rastreamento_kpis', 'squad1.gold_rastreamento_sla', 'squad1.gold_rastreamento_por_transportadora']

In [0]:
# Validação final das tabelas criadas
for tabela in tabelas_gold_criadas:
    print(f"\n{tabela}")
    spark.sql(f"DESCRIBE DETAIL {tabela}").select("format", "numFiles", "sizeInBytes", "location").show(truncate=False)
    print(f"Registros: {spark.table(tabela).count()}")
    display(spark.table(tabela).limit(10))
